# 05 Research Lineage and Version Comparison

Inspect the full research chain and compare two saved runs without losing their signal, config, parent-run, or code-version context.

In [ ]:
from pathlib import Path
import sys

def find_project_root():
    candidates = [Path.cwd().resolve(), Path('Z:/SEN05_Autotrading'), Path('//10.11.12.6/Share/SEN05_Autotrading')]
    for candidate in candidates:
        current = candidate
        while True:
            if (current / 'pyproject.toml').exists() and (current / 'backtest_optimize').exists():
                return current
            if current.parent == current:
                break
            current = current.parent
    raise RuntimeError('Could not find project root.')

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
SNAPSHOT_DIR = project_root / 'backtest_optimize' / 'outputs' / 'version_snapshots'

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from backtest_optimize.analysis.versioning import load_snapshot

pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 260)

In [ ]:
files = sorted(SNAPSHOT_DIR.glob('*.json'), key=lambda path: path.stat().st_mtime, reverse=True)
records = []
for path in files:
    record = load_snapshot(path)
    record['snapshot_path'] = str(path)
    records.append(record)
if not records:
    raise FileNotFoundError('No snapshots found. Run notebooks 01-04 first.')

df = pd.json_normalize(records, sep='.')
lineage_columns = [
    'created_at', 'run_type', 'run_id', 'parent_run_id', 'name',
    'config.strategy', 'config.symbol', 'config.timeframe',
    'git_commit', 'signal_file_md5', 'config_hash', 'snapshot_path',
]
available_lineage = [column for column in lineage_columns if column in df.columns]
display(Markdown('## Research Lineage'))
display(df[available_lineage].head(100))

type_summary = (
    df.assign(run_type=df.get('run_type', pd.Series('legacy', index=df.index)).fillna('legacy'))
      .groupby('run_type', dropna=False)
      .size()
      .rename('run_count')
      .reset_index()
)
display(Markdown('## Runs by Stage'))
display(type_summary)

In [ ]:
# Select two rows from the lineage table for a field-by-field comparison.
LEFT_INDEX = 0
RIGHT_INDEX = 1 if len(df) > 1 else 0

left = df.iloc[LEFT_INDEX]
right = df.iloc[RIGHT_INDEX]
diff_rows = []
for column in sorted(df.columns):
    left_value = left.get(column)
    right_value = right.get(column)
    if str(left_value) != str(right_value):
        diff_rows.append({'field': column, 'left': left_value, 'right': right_value})
diff = pd.DataFrame(diff_rows)

display(Markdown('## Selected Runs'))
display(pd.DataFrame([
    {'side': 'left', 'index': LEFT_INDEX, 'run_id': left.get('run_id', left.get('name')), 'run_type': left.get('run_type')},
    {'side': 'right', 'index': RIGHT_INDEX, 'run_id': right.get('run_id', right.get('name')), 'run_type': right.get('run_type')},
]))
display(Markdown('## Differences'))
display(diff.head(200))

In [ ]:
# Optional: inspect one complete lineage chain by starting from a child run.
START_RUN_ID = df.iloc[LEFT_INDEX].get('run_id') or df.iloc[LEFT_INDEX].get('name')
by_run_id = {
    (record.get('run_id') or record.get('name')): record
    for record in records
    if record.get('run_id') or record.get('name')
}
chain = []
current_id = START_RUN_ID
visited = set()
while current_id and current_id not in visited:
    visited.add(current_id)
    record = by_run_id.get(current_id)
    if record is None:
        break
    chain.append({
        'run_id': current_id,
        'run_type': record.get('run_type'),
        'parent_run_id': record.get('parent_run_id'),
        'created_at': record.get('created_at'),
        'config_hash': record.get('config_hash'),
    })
    current_id = record.get('parent_run_id')
display(Markdown('## Parent Chain'))
display(pd.DataFrame(chain))